In [2]:
from pathlib import Path

BASE_PATH = Path('/oak/stanford/groups/ckeller1/data/iEEG_EHR/iEEG_NWB')

# Get all files for one subject/session
sub_id = 'sub-231'
ses_id = 'ses-01'

raw_files = sorted(BASE_PATH.glob(f'{sub_id}/{ses_id}/ieeg/*.nwb'))
prep_files = sorted(BASE_PATH.glob(f'{sub_id}/{ses_id}/preprocessed/*_bipolar_psd.nwb'))

print(f"Subject: {sub_id}, Session: {ses_id}")
print(f"Raw files: {len(raw_files)}")
print(f"Preprocessed files: {len(prep_files)}")

# Extract run IDs from both
def get_run_id(filename):
    parts = filename.split('_')
    for part in parts:
        if part.startswith('run-'):
            return part.replace('.nwb', '').replace('_bipolar_psd.nwb', '')
    return None

raw_run_ids = {get_run_id(f.name): f.name for f in raw_files}
prep_run_ids = {get_run_id(f.name): f.name for f in prep_files}

print(f"\nRaw run IDs (first 10): {list(raw_run_ids.keys())[:10]}")
print(f"\nPreprocessed run IDs (first 10): {list(prep_run_ids.keys())[:10]}")

# Check for matches
matches = set(raw_run_ids.keys()) & set(prep_run_ids.keys())
raw_only = set(raw_run_ids.keys()) - set(prep_run_ids.keys())
prep_only = set(prep_run_ids.keys()) - set(raw_run_ids.keys())

print(f"\n{'='*60}")
print(f"Matching analysis:")
print(f"{'='*60}")
print(f"Runs with both raw and preprocessed: {len(matches)}")
print(f"Runs with only raw: {len(raw_only)}")
print(f"Runs with only preprocessed: {len(prep_only)}")

if len(matches) > 0:
    print(f"\nExample matches:")
    for run_id in list(matches)[:5]:
        print(f"  {run_id}")
        print(f"    Raw: {raw_run_ids[run_id]}")
        print(f"    Prep: {prep_run_ids[run_id]}")

if len(raw_only) > 0:
    print(f"\nExamples with only raw (not preprocessed):")
    for run_id in list(raw_only)[:5]:
        print(f"  {run_id}: {raw_run_ids[run_id]}")

if len(prep_only) > 0:
    print(f"\nExamples with only preprocessed (no raw):")
    for run_id in list(prep_only)[:5]:
        print(f"  {run_id}: {prep_run_ids[run_id]}")

Subject: sub-231, Session: ses-01
Raw files: 168
Preprocessed files: 153

Raw run IDs (first 10): ['run-EA63018F', 'run-EA63018G', 'run-EA63018H', 'run-EA63018I', 'run-EA63018J', 'run-EA63018K', 'run-EA63018L', 'run-EA63018M', 'run-EA63018N', 'run-EA63018O']

Preprocessed run IDs (first 10): ['run-EA63018F', 'run-EA63018G', 'run-EA63018H', 'run-EA63018I', 'run-EA63018J', 'run-EA63018K', 'run-EA63018L', 'run-EA63018M', 'run-EA63018N', 'run-EA63018O']

Matching analysis:
Runs with both raw and preprocessed: 153
Runs with only raw: 15
Runs with only preprocessed: 0

Example matches:
  run-EA63019M
    Raw: sub-231_ses-01_run-EA63019M.nwb
    Prep: sub-231_ses-01_run-EA63019M_bipolar_psd.nwb
  run-EA6301CI
    Raw: sub-231_ses-01_run-EA6301CI.nwb
    Prep: sub-231_ses-01_run-EA6301CI_bipolar_psd.nwb
  run-EA63019S
    Raw: sub-231_ses-01_run-EA63019S.nwb
    Prep: sub-231_ses-01_run-EA63019S_bipolar_psd.nwb
  run-EA6301CU
    Raw: sub-231_ses-01_run-EA6301CU.nwb
    Prep: sub-231_ses-01_ru

In [ ]:


if __name__ == '__main__':
    import argparse
    
    parser = argparse.ArgumentParser(description='Create NWB metadata CSV with file creation times')
    parser.add_argument('--output', type=str, default='nwb_metadata.csv',
                       help='Output CSV filename')
    parser.add_argument('--analyze-dates', action='store_true',
                       help='Print file creation date summary after creating metadata')
    parser.add_argument('--cutoff-date', type=str, default='2025-02-08',
                       help='Cutoff date for old file analysis (YYYY-MM-DD)')
    
    args = parser.parse_args()
    
    # Create metadata
    df = create_metadata_csv(args.output)
    
    # Optionally analyze dates
    if args.analyze_dates:
        print_creation_date_summary(df)
        analyze_old_files(df, args.cutoff_date)